In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

ea_dir = r"C:\Users\user\Downloads\GSE148812_clean"

# Reload everything needed
gene_burden_ea = pd.read_csv(os.path.join(ea_dir, "gene_burden_matrix_signed_protein_coding_EA.csv"), index_col=0)
results_df_ea = pd.read_csv(os.path.join(ea_dir, "gene_doubleml_stability_smoking_EA.csv"))
shortlist_100_ea = results_df_ea[results_df_ea["stability_fraction"] == 1.0].copy()

meta_df = pd.read_csv(os.path.join(ea_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")
meta_df["smoking_status_bin"] = (meta_df["smoking_status"] == "Smoker").astype(int)

sample_cols_ea = gene_burden_ea.columns.tolist()
pheno_ea = meta_df.loc[sample_cols_ea, "smoking_status_bin"].astype(float)

# Check for near-duplicate genes first (r^2 > 0.99), same check as AA
gene_burden_shortlist_ea = gene_burden_ea.loc[shortlist_100_ea["gene"]]
corr_ea = gene_burden_shortlist_ea.T.corr()
r2_ea = (corr_ea ** 2).values.copy()
np.fill_diagonal(r2_ea, 0)
dup_pairs_ea = np.argwhere(r2_ea > 0.99)
dup_genes_to_drop = set()
for i, j in dup_pairs_ea:
    if i < j:
        g1 = shortlist_100_ea["gene"].iloc[i]
        g2 = shortlist_100_ea["gene"].iloc[j]
        print(f"Near-duplicate: {g1} <-> {g2}, r²={r2_ea[i,j]:.4f}")
        dup_genes_to_drop.add(g2)  # drop the second of each pair

shortlist_dedup_ea = shortlist_100_ea[~shortlist_100_ea["gene"].isin(dup_genes_to_drop)]
print(f"\nEA genes after dedup: {len(shortlist_dedup_ea)} (dropped {len(dup_genes_to_drop)})")

genes_list_ea = shortlist_dedup_ea.sort_values("stability_fraction", ascending=False)["gene"].tolist()

gene_burden_final_ea = gene_burden_ea.loc[genes_list_ea]
X_genes_pc_ea = gene_burden_final_ea.T.values
Y_pc_ea = pheno_ea.values.reshape(-1, 1)
X_pc_full_ea = np.hstack([X_genes_pc_ea, Y_pc_ea])
col_names_ea = gene_burden_final_ea.index.tolist() + ["smoking_status"]

print("EA PC input shape:", X_pc_full_ea.shape)

# 2-group split (validated method from AA)
n_groups_2 = 2
groups_ea_2 = [genes_list_ea[i::n_groups_2] for i in range(n_groups_2)]
for idx, g in enumerate(groups_ea_2):
    print(f"Group {idx+1}: {len(g)} genes")

all_direct_parents_ea = {}
for group_idx, gene_group in enumerate(groups_ea_2):
    group_cols = gene_group + ["smoking_status"]
    idx_subset = [col_names_ea.index(c) for c in group_cols]
    X_subset = X_pc_full_ea[:, idx_subset]

    n = len(group_cols)
    outcome_idx = group_cols.index("smoking_status")

    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in group_cols]
    for i in range(n - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    start = time.time()
    cg = pc(
        data=X_subset,
        alpha=0.001,
        indep_test=fisherz,
        stable=True,
        uc_rule=0,
        uc_priority=2,
        background_knowledge=bk,
        depth=3,
        verbose=False,
        show_progress=False,
        node_names=group_cols
    )
    elapsed = time.time() - start
    print(f"Group {group_idx+1} ({n} nodes): {elapsed:.1f}s")

    adj = cg.G.graph
    direct_parents = [group_cols[i] for i in range(n) if group_cols[i] != "smoking_status"
                       and ((adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == 1)
                            or (adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == -1))]
    print(f"  Direct parents: {direct_parents}")
    all_direct_parents_ea[f"group_{group_idx+1}"] = direct_parents

total_ea_parents = [g for parents in all_direct_parents_ea.values() for g in parents]
print(f"\nTotal EA direct parents: {len(total_ea_parents)}")
print(total_ea_parents)

# Check the 3 candidate genes specifically
candidates = ['DNAH1', 'FRAS1', 'HYDIN']
print("\nOf DNAH1/FRAS1/HYDIN, survived EA PC:", [g for g in candidates if g in total_ea_parents])

c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



EA genes after dedup: 62 (dropped 0)
EA PC input shape: (1460, 63)
Group 1: 31 genes
Group 2: 31 genes
Group 1 (32 nodes): 1.7s
  Direct parents: ['ABCA8', 'MYBBP1A', 'SMG5', 'SYNE1', 'DICER1', 'PPARGC1B', 'DNAH10', 'DCHS1', 'RP11-1055B8.7', 'OBSL1', 'ATP12A']
Group 2 (32 nodes): 5.6s
  Direct parents: ['KMT2A', 'ZFHX4', 'LRP1B', 'SULT1C2', 'FRYL', 'SACS', 'DNAH1', 'PRKDC', 'MAGI1', 'CP', 'OTOF', 'ATP2C2', 'MLLT1']

Total EA direct parents: 24
['ABCA8', 'MYBBP1A', 'SMG5', 'SYNE1', 'DICER1', 'PPARGC1B', 'DNAH10', 'DCHS1', 'RP11-1055B8.7', 'OBSL1', 'ATP12A', 'KMT2A', 'ZFHX4', 'LRP1B', 'SULT1C2', 'FRYL', 'SACS', 'DNAH1', 'PRKDC', 'MAGI1', 'CP', 'OTOF', 'ATP2C2', 'MLLT1']

Of DNAH1/FRAS1/HYDIN, survived EA PC: ['DNAH1']


In [2]:
print("Group 1 direct parent count:", len(all_direct_parents_ea['group_1']))
print("Group 2 direct parent count:", len(all_direct_parents_ea['group_2']))
print("Total genes tested:", 62)
print("Fraction direct parents:", 24/62)

Group 1 direct parent count: 11
Group 2 direct parent count: 13
Total genes tested: 62
Fraction direct parents: 0.3870967741935484


In [3]:
import random
random.seed(99)
genes_shuffled_ea = genes_list_ea.copy()
random.shuffle(genes_shuffled_ea)
groups_ea_v2 = [genes_shuffled_ea[i::2] for i in range(2)]

all_direct_parents_ea_v2 = {}
for group_idx, gene_group in enumerate(groups_ea_v2):
    group_cols = gene_group + ["smoking_status"]
    idx_subset = [col_names_ea.index(c) for c in group_cols]
    X_subset = X_pc_full_ea[:, idx_subset]
    n = len(group_cols)
    outcome_idx = group_cols.index("smoking_status")

    bk = BackgroundKnowledge()
    nodes = [GraphNode(name) for name in group_cols]
    for i in range(n - 1):
        bk.add_node_to_tier(nodes[i], 0)
    bk.add_node_to_tier(nodes[outcome_idx], 1)

    cg = pc(data=X_subset, alpha=0.001, indep_test=fisherz, stable=True,
            uc_rule=0, uc_priority=2, background_knowledge=bk, depth=3,
            verbose=False, show_progress=False, node_names=group_cols)

    adj = cg.G.graph
    direct_parents = [group_cols[i] for i in range(n) if group_cols[i] != "smoking_status"
                       and ((adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == 1)
                            or (adj[i, outcome_idx] == -1 and adj[outcome_idx, i] == -1))]
    all_direct_parents_ea_v2[f"group_{group_idx+1}"] = direct_parents

total_ea_v2 = [g for parents in all_direct_parents_ea_v2.values() for g in parents]
print("Total direct parents (2nd random split):", len(total_ea_v2))

overlap_ea = set(total_ea_parents) & set(total_ea_v2)
print(f"\nOverlap between the two EA splits: {len(overlap_ea)} / {len(set(total_ea_parents) | set(total_ea_v2))}")
print("Stable genes:", sorted(overlap_ea))

# Check our candidates specifically
print("\nDNAH1 stable across both EA splits:", "DNAH1" in overlap_ea)
print("CP stable across both EA splits:", "CP" in overlap_ea)

Total direct parents (2nd random split): 23

Overlap between the two EA splits: 19 / 28
Stable genes: ['ABCA8', 'ATP12A', 'CP', 'DCHS1', 'DICER1', 'DNAH1', 'DNAH10', 'FRYL', 'KMT2A', 'LRP1B', 'MAGI1', 'MLLT1', 'MYBBP1A', 'OBSL1', 'OTOF', 'PPARGC1B', 'PRKDC', 'RP11-1055B8.7', 'SACS']

DNAH1 stable across both EA splits: True
CP stable across both EA splits: True
